### Import the libraries

In [90]:
import spacy
from spacy import tokenizer
from bs4 import BeautifulSoup
import nltk
import string
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize.toktok import ToktokTokenizer
import pandas as pd

### Load the covid19 tweet data

In [40]:
# Read the data from the Excel and load into the dataframe using pandas
rawData =pd.read_excel("COVID_19_vaccine_100.xlsx")
rawData.columns=['tweets']
rawData.head(5)

,tweets
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...
1,May i remind you that the vaccine isnt just fo...
2,"@MJAckermanMDPhD\nHi, Dr. Ackerman. I will lik..."
3,"Shandro, Hinshaw to give COVID-19 vaccine upda..."
4,You ever Noticed This? They just announced Yes...


### NULL Check

In [41]:
# Check nulls and if it hampers processing drop them
rawData['tweets'].isnull().sum()

0

## Text Processing

#### Removing html tags

<span style="color:white">Often, unstructured text contains a lot of noise, especially if you use techniques like web or screen scraping. HTML tags are typically one of these components which don’t add much value towards understanding and analyzing text.</span>

In [42]:
def strip_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    stripped_text = soup.get_text()
    return stripped_text
strip_html_tags('<html><h2>May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO</h2></html>')

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

#### Removing accented characters
<span style="color:white">Usually in any text corpus, you might be dealing with accented characters/letters, especially if you only want to analyze the English language. Hence, we need to make sure that these characters are converted and standardized into ASCII characters. A simple example — converting é to e.</span>

In [43]:
import unicodedata


def remove_accented_chars(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text

remove_accented_chars('Sómě Áccěntěd těxt')
remove_accented_chars("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO'

### Contraction MAP

In [81]:
#Contraction Mapping
CONTRACTION_MAP = {
"ain't": "is not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he'll've": "he he will have",
"he's": "he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how is",
"I'd": "I would",
"I'd've": "I would have",
"I'll": "I will",
"I'll've": "I will have",
"I'm": "I am",
"I've": "I have",
"i'd": "i would",
"i'd've": "i would have",
"i'll": "i will",
"i'll've": "i will have",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'd've": "it would have",
"it'll": "it will",
"it'll've": "it will have",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she would",
"she'd've": "she would have",
"she'll": "she will",
"she'll've": "she will have",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so as",
"that'd": "that would",
"that'd've": "that would have",
"that's": "that is",
"there'd": "there would",
"there'd've": "there would have",
"there's": "there is",
"they'd": "they would",
"they'd've": "they would have",
"they'll": "they will",
"they'll've": "they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what'll've": "what will have",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"when's": "when is",
"when've": "when have",
"where'd": "where did",
"where's": "where is",
"where've": "where have",
"who'll": "who will",
"who'll've": "who will have",
"who's": "who is",
"who've": "who have",
"why's": "why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you would",
"you'd've": "you would have",
"you'll": "you will",
"you'll've": "you will have",
"you're": "you are",
"you've": "you have"
}

#### Expanding Contractions
<span style="color:white">Contractions are shortened version of words or syllables. They often exist in either written or spoken forms in the English language. These shortened versions or contractions of words are created by removing specific letters and sounds. In case of English contractions, they are often created by removing one of the vowels from the word. Examples would be, do not to don’t and I would to I’d. Converting each contraction to its expanded, original form helps with text standardization.</span>

In [83]:
def expand_contractions(text, contraction_mapping=CONTRACTION_MAP):
    modified_text=[]
    for words in text.split(' '):
        val = CONTRACTION_MAP.get(words)
        if val is not None:
            modified_text.append(val)
        else:
            modified_text.append(words)
    modified_text=' '.join(modified_text)
    return modified_text

expand_contractions("Y'all are enjoying this class we'd think. It is so cool isnt it?")

"Y'all are enjoying this class we would think. It is so cool isnt it?"

#### Removing Special Characters
<span style="color:white">Special characters and symbols are usually non-alphanumeric characters or even occasionally numeric characters (depending on the problem), which add to the extra noise in unstructured text. Usually, simple regular expressions (regexes) can be used to remove them.</span>

In [84]:
def remove_special_characters(text, remove_digits=True):
    pattern = r'[^a-zA-z0-9\s]' if not remove_digits else r'[^a-zA-z\s]'
    text = re.sub(pattern, '', text)
    return text

remove_special_characters("@johensley @darcyshepherd13 @jeffreyguterman @realdonaldtrump the different vaccines have different ingredients. i will take a covid-19 in a second. 95% efficacy is fine by me. also, thevaccine is not made out of recombinant dna, it is made out of mrna. i really hate when people spread misinformation. luciferase is an enzyme. not in vaccines", remove_digits=True)
#remove_special_characters("May i remind you that the vaccine isnt just for those who @caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'johensley darcyshepherd jeffreyguterman realdonaldtrump the different vaccines have different ingredients i will take a covid in a second  efficacy is fine by me also thevaccine is not made out of recombinant dna it is made out of mrna i really hate when people spread misinformation luciferase is an enzyme not in vaccines'

#### Removing Stopwords
<span style="color:white">Words which have little or no significance, especially when constructing meaningful features from text, are known as stopwords or stop words. These are usually words that end up having the maximum frequency if you do a simple term or word frequency in a corpus. Typically, these can be articles, conjunctions, prepositions and so on. Some examples of stopwords are a, an, the, and the like.</span>

In [85]:

stopword_list = nltk.corpus.stopwords.words('english')
tokenizer = ToktokTokenizer()
def remove_stopwords(text, is_lower_case=False):
    tokens = tokenizer.tokenize(text)
    tokens = [token.strip() for token in tokens]
    if is_lower_case:
        filtered_tokens = [token for token in tokens if token not in stopword_list]
    else:
        filtered_tokens = [token for token in tokens if token.lower() not in stopword_list]
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

remove_stopwords("Let us see if we can or can not remove against the stopwords from a sentence.")
remove_stopwords("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO")

'May remind vaccine isnt caught covid-19 , also prevent ................ https://t.co/htFc4jP4CO'

#### Remomving URLS
<span style="color:white">Words which contains urls are mostly not required for the data analysis</span>

In [86]:
def url_removal(text):
    modified_text = ' '.join([words for words in text.split(' ') if words[0:4]!="http"])
    return modified_text

url_removal("May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................ https://t.co/htFc4jP4CO" )


'May i remind you that the vaccine isnt just for those who caught covid-19, its also to prevent it................'

### Lemmatization

In [87]:
wl =WordNetLemmatizer()
def lemmatize_text(text):
    text = ' '.join([wl.lemmatize(i) for i in text.split(' ')])
    return text


### Stemming

In [88]:
ps = nltk.PorterStemmer()
def stem_text(text):
    text = ' '.join([ps.stem(i) for i in text.split(' ')])
    return text

## Normalization

In [89]:
#Wrapping all the above processes
def normalize_corpus(doc, html_stripping=True, contraction_expansion=True,
                     accented_char_removal=True, text_lower_case=True, special_char_removal=True,
                     stopword_removal=True,remove_url=True, lemmatize=True,stem=True,remove_digits=True):

    normalized_corpus = []
    if html_stripping:
        doc= strip_html_tags(doc)
    # remove accented characters
    if accented_char_removal:
        doc = remove_accented_chars(doc)
    # expand contractions
    if contraction_expansion:
        doc = expand_contractions(doc)
    # lowercase the text
    if text_lower_case:
        doc = doc.lower()
    # # remove extra newlines
    # doc = re.sub(r'[\r|\n|\r\n]+', ' ',doc)
    # remove special characters and\or digits
    if special_char_removal:
        doc=remove_special_characters(doc,remove_digits=True)
    #  # remove extra whitespace
    # doc = re.sub(' +', ' ', doc)
    # remove stopwords
    if stopword_removal:
        doc = remove_stopwords(doc, is_lower_case=text_lower_case)
    if remove_url:
        doc=url_removal(doc)
    #lemmatize text
    if lemmatize:
        doc = lemmatize_text(doc)
    # #stemming text
    # if stem:
    #     doc = stem_text(doc)
    normalized_corpus.append(doc)
    return ' '.join(normalized_corpus)

rawData['tweets_cleaned']= rawData['tweets'].apply(lambda x:normalize_corpus(x))
rawData['tweets_cleaned']

C:\Users\ragha\AppData\Local\Temp\ipykernel_9112\172976477.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


0     johensley darcyshepherd jeffreyguterman realdo...
1     may remind vaccine isnt caught covid also prevent
2     mjackermanmdphd hi dr ackerman like know going...
3        shandro hinshaw give covid vaccine update noon
4     ever noticed announced yesterday december th g...
                            ...                        
95    must viewing principle vaccine program control...
96    covid vaccine protection virus outweighs poten...
97    spent time today looking whether covid vaccine...
98    covid vaccine likely beneficial breastfed baby...
99    alright guy want make trip pact cuz iatmm summ...
Name: tweets_cleaned, Length: 100, dtype: object

### POS Tagging

In [51]:
import spacy
nltk.download('averaged_perceptron_tagger')
# Download NLTK Punkt sentence tokenizer
nltk.download('punkt')
nlp = spacy.load("en_core_web_sm")
pos_tagged_data=[]
for sentence in rawData['tweets_cleaned']:
    # print(sentence)
    sentence_nlp = nlp(sentence)
    # print(sentence_nlp,end='\n')
    spacy_pos_tagged=[]
    spacy_pos_tagged_sentence = [(word.__str__().strip(), word.tag_, word.pos_,sentence) for word in [i for i in sentence_nlp]]
    pos_tagged_data=pos_tagged_data+spacy_pos_tagged_sentence


spacy_pos_tagged_data = pd.DataFrame(pos_tagged_data,columns=['word','pos_tag','tag_type','sentence'])
spacy_pos_tagged_data.to_csv('spacy_pos_tagged_data.csv')
spacy_pos_tagged_data.head(20)


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ragha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,word,pos_tag,tag_type,sentence
0,johensley,NNP,PROPN,johensley darcyshepherd jeffreyguterman realdo...
1,darcyshepherd,NNP,PROPN,johensley darcyshepherd jeffreyguterman realdo...
2,jeffreyguterman,NNP,PROPN,johensley darcyshepherd jeffreyguterman realdo...
3,realdonaldtrump,VB,VERB,johensley darcyshepherd jeffreyguterman realdo...
4,different,JJ,ADJ,johensley darcyshepherd jeffreyguterman realdo...
5,vaccine,NN,NOUN,johensley darcyshepherd jeffreyguterman realdo...
6,different,JJ,ADJ,johensley darcyshepherd jeffreyguterman realdo...
7,ingredient,NN,NOUN,johensley darcyshepherd jeffreyguterman realdo...
8,take,VB,VERB,johensley darcyshepherd jeffreyguterman realdo...
9,covid,JJ,ADJ,johensley darcyshepherd jeffreyguterman realdo...


## Question 1:

### 1A Frequency of the word covid appears in the dataset:

In [52]:
print("Frequency of the word 'covid' is: {}".format(spacy_pos_tagged_data[spacy_pos_tagged_data['word']=='covid']['word'].count()))
spacy_pos_tagged_data[spacy_pos_tagged_data['word']=='covid']

Frequency of the word 'covid' is: 111


,word,pos_tag,tag_type,sentence
9,covid,JJ,ADJ,johensley darcyshepherd jeffreyguterman realdo...
34,covid,JJ,ADJ,may remind vaccine isnt caught covid also prevent
46,covid,JJ,ADJ,mjackermanmdphd hi dr ackerman like know going...
53,covid,JJ,ADJ,shandro hinshaw give covid vaccine update noon
66,covid,JJ,ADJ,ever noticed announced yesterday december th g...
...,...,...,...,...
1561,covid,JJ,ADJ,must viewing principle vaccine program control...
1564,covid,NNP,PROPN,covid vaccine protection virus outweighs poten...
1580,covid,JJ,ADJ,spent time today looking whether covid vaccine...
1601,covid,JJ,ADJ,covid vaccine likely beneficial breastfed baby...


### 1B Word count of 'covid' across each pos_tag type

In [53]:
covid_word_pos_tags = spacy_pos_tagged_data[spacy_pos_tagged_data['word']=='covid'].groupby(
    ['word','pos_tag']).size().reset_index(name='counts').sort_values('counts',ascending=False)
covid_word_pos_tags.to_csv('covid_word_pos_tags.csv')
covid_word_pos_tags

,word,pos_tag,counts
0,covid,JJ,61
2,covid,NNP,43
1,covid,NN,5
3,covid,VB,2


In [54]:
covid_word_sntns_pos_tags = spacy_pos_tagged_data[spacy_pos_tagged_data['word']=='covid'].groupby(
    ['word','sentence','pos_tag']).size().reset_index(name='counts')
covid_word_sntns_pos_tags.to_csv('covid_word_sntns_pos_tags.csv')
covid_word_sntns_pos_tags.head(10)

,word,sentence,pos_tag,counts
0,covid,alright guy want make trip pact cuz iatmm summ...,NNP,1
1,covid,amoderna expects covid vaccine protect uk coro...,JJ,1
2,covid,ano covid related death covid case requiring m...,NN,1
3,covid,ano covid related death covid case requiring m...,NNP,1
4,covid,asked answered check updated north carolina co...,NNP,1
5,covid,az scheint gut zu wirken afirst dose bntb vacc...,JJ,1
6,covid,b poll plan receiving type covid vaccine covid...,JJ,1
7,covid,baby vaccination new study suggests pfizers co...,JJ,1
8,covid,bell palsy conventional medicine say cause unk...,JJ,1
9,covid,bob_wachter peter mark explained changing dose...,NN,1


### Question 2:

## 2 Cardinal entity(CD) Count

In [55]:
pos_cardinal_count = spacy_pos_tagged_data[spacy_pos_tagged_data['pos_tag']=='CD'].groupby(["word","pos_tag"]).size().reset_index(name='counts').sort_values('counts',ascending=False)
pos_cardinal_count.to_csv('cardinal_counts.csv')
pos_cardinal_count.head(10)

,word,pos_tag,counts
2,one,CD,5
4,two,CD,4
0,hundred,CD,1
1,million,CD,1
3,six,CD,1


In [56]:
spacy_pos_tagged_data[spacy_pos_tagged_data["pos_tag"]=='CD'].groupby(["word","pos_tag",'sentence']).size().reset_index(name='counts').sort_values('counts', ascending=False)

,word,pos_tag,sentence,counts
0,hundred,CD,coronavaccine govt want people trust vaccine e...,1
1,million,CD,vernersviews anyone providing number people ma...,1
2,one,CD,californiaatms state epidemiologist urged covi...,1
3,one,CD,davekeating mentioned earlier tweet learn covi...,1
4,one,CD,novavax say covid vaccine effective far le one...,1
5,one,CD,rrrbyn covid vaccine use one two human fetal c...,1
6,one,CD,youatmre looking thorough easytounderstand exp...,1
7,six,CD,december united state seen six case anaphylaxi...,1
8,two,CD,dad demand answer health worker daughter died ...,1
9,two,CD,december united state seen six case anaphylaxi...,1


### Question 3
<span color:white>Identify the top 5 most commonly appearing entity types in the provided dataset and determine
their respective frequencies using the Spacy Named Entity Recognition (NER).</span>

### NER

### Five most commonly appearing entity types

In [58]:
ner_tags=[]
for i in rawData['tweets_cleaned']:
    doc = nlp(i)
    if doc.ents:
        ner_tags= ner_tags+[(ent.text,ent.label_,word.tag_) for word,ent in zip(doc,doc.ents)]
    # else:
    #     print(i)

ner_tags_data =pd.DataFrame(ner_tags,columns=['text','label','pos_tag'])
ner_tags_data.head(200)

,text,label,pos_tag
0,johensley darcyshepherd jeffreyguterman,PERSON,NNP
1,second,ORDINAL,NNP
2,mrna,GPE,NNP
3,dr ackerman,PERSON,NNP
4,shandro hinshaw,PERSON,NNP
...,...,...,...
105,californiaatms state,ORG,NNP
106,one,CARDINAL,NNP
107,covid vaccine protection,PERSON,NNP
108,today,DATE,VBD


In [20]:
ner_tags_data.groupby(['label','pos_tag']).size().reset_index(name='count').sort_values('count',ascending=False)

,label,pos_tag,count
30,PERSON,NNP,23
24,ORG,NNP,12
28,PERSON,JJ,7
4,DATE,NNP,6
1,CARDINAL,NNP,6
23,ORG,NN,4
12,GPE,NN,4
19,NORP,NNP,3
20,ORDINAL,NNP,3
13,GPE,NNP,3


### Question 4

### Sentiment Analysis

In [21]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [22]:
sid = SentimentIntensityAnalyzer()

In [23]:
sid.polarity_scores(rawData['tweets_cleaned'][0])

{'neg': 0.19, 'neu': 0.755, 'pos': 0.054, 'compound': -0.6697}

### Adding new scores column to store scores predicted by sid

In [24]:
rawData['scores'] = rawData['tweets_cleaned'].apply(lambda x:sid.polarity_scores(x))
rawData['scores']

0     {'neg': 0.19, 'neu': 0.755, 'pos': 0.054, 'com...
1     {'neg': 0.0, 'neu': 0.864, 'pos': 0.136, 'comp...
2     {'neg': 0.0, 'neu': 0.671, 'pos': 0.329, 'comp...
3     {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
4     {'neg': 0.146, 'neu': 0.53, 'pos': 0.325, 'com...
                            ...                        
95    {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...
96    {'neg': 0.174, 'neu': 0.826, 'pos': 0.0, 'comp...
97    {'neg': 0.193, 'neu': 0.807, 'pos': 0.0, 'comp...
98    {'neg': 0.0, 'neu': 0.756, 'pos': 0.244, 'comp...
99    {'neg': 0.128, 'neu': 0.617, 'pos': 0.255, 'co...
Name: scores, Length: 100, dtype: object

### Adding new compund column to store either positive or negative value of the scores

In [25]:
rawData['compound'] = rawData['scores'].apply(lambda x : 1 if x['compound']>0 else -1)
rawData['compound']

0    -1
1     1
2     1
3    -1
4     1
     ..
95   -1
96   -1
97   -1
98    1
99    1
Name: compound, Length: 100, dtype: int64

### Total positive and negative counts

In [26]:
print('Positive Tweets: {}, Negative Tweets: {}'.format(rawData[rawData['compound']==1]['compound'].count(),rawData[rawData['compound']==-1]['compound'].count()))

Positive Tweets: 40, Negative Tweets: 60


### Topic Modelling

In [59]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [60]:
cv = CountVectorizer(max_df=0.95,min_df=2,stop_words='english')

In [61]:
dtm = cv.fit_transform(rawData['tweets_cleaned'])
dtm

<100x165 sparse matrix of type '<class 'numpy.int64'>'
	with 507 stored elements in Compressed Sparse Row format>

### Topic Modelling using LDA

In [62]:
LDA = LatentDirichletAllocation(n_components=5,random_state=42)
LDA.fit(dtm)
tranformed_data = LDA.transform(dtm)

In [63]:
cv.get_feature_names_out()[50]

'expert'

In [64]:
LDA.components_.shape

(5, 165)

In [35]:
for i,topic in enumerate(LDA.components_):
    print('Top word in Topic: {}'.format(i),[cv.get_feature_names_out()[index] for index in topic.argsort()[-1:]])

Top word in Topic: 0 ['coronavirus']
Top word in Topic: 1 ['efficacy']
Top word in Topic: 2 ['concern']
Top word in Topic: 3 ['safe']
Top word in Topic: 4 ['effect']


In [36]:
rawData['topic']=tranformed_data.argmax(axis=1)

In [37]:
rawData[rawData['topic']==0 & rawData['tweets_cleaned'].str.contains('coronavirus')]

,tweets,tweets_cleaned,scores,compound,topic
0,@johensley @DarcyShepherd13 @JeffreyGuterman @...,johensley darcyshepherd jeffreyguterman realdo...,"{'neg': 0.19, 'neu': 0.755, 'pos': 0.054, 'com...",-1,0
18,Can a COVID-19 vaccine alone end the pandemic?...,covid vaccine alone end pandemic take question...,"{'neg': 0.143, 'neu': 0.857, 'pos': 0.0, 'comp...",-1,0
19,Novavax publishes positive efficacy data for i...,novavax publishes positive efficacy data covid...,"{'neg': 0.0, 'neu': 0.66, 'pos': 0.34, 'compou...",1,0
20,""" #Covid_19\n**single vaccine dose** leads to ...",covid_ single vaccine dose lead greater risk n...,"{'neg': 0.194, 'neu': 0.667, 'pos': 0.139, 'co...",-1,0
25,Think twice before you get that COVID-19 Vacci...,think twice get covid vaccine,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",-1,0
27,Nice infographic on COVID-19 vaccines and how ...,nice infographic covid vaccine compare unbiase...,"{'neg': 0.0, 'neu': 0.641, 'pos': 0.359, 'comp...",1,0
32,All data pertaining to trials should be made p...,data pertaining trial made public expert covid...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",-1,0
36,@VernersViews Anyone providing numbers of peop...,vernersviews anyone providing number people ma...,"{'neg': 0.223, 'neu': 0.557, 'pos': 0.22, 'com...",-1,0
43,#CoronaVaccine And this govt wants people to t...,coronavaccine govt want people trust vaccine e...,"{'neg': 0.162, 'neu': 0.631, 'pos': 0.207, 'co...",-1,0
50,"Dr. Kevin Most on COVID-19 vaccine, another st...",dr kevin covid vaccine another strain coronavi...,"{'neg': 0.146, 'neu': 0.854, 'pos': 0.0, 'comp...",-1,0


### Topic1 Sentiment

In [80]:
topic1=LDA.components_[0]
# for i,j in zip(single_topic.argsort(),single_topic):
#     print(cv.get_feature_names_out()[i],j)
pos = 0
neg = 0
for i in topic1.argsort():
    dict = sid.polarity_scores(cv.get_feature_names_out()[i])
    if dict['compound']>0:
        pos=pos+1
    else:
        neg = neg+1

if pos>neg:
    print("Topic 1 has Positive Sentiment")
else:
    print("Topic 1 has Negative Sentiment")

Topic 1 has Negative Sentiment
